# 20.9 A/B testing and online experimentation

Online experiments estimate causal product impact by comparing randomized traffic, not by trusting before-after movement.

Serving supplies traffic splitting and monitoring supplies guardrails. This notebook computes pooled standard errors, z statistics, and peeking behavior across experiment workloads.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2009
rng = np.random.default_rng(SEED)


def ab_test_z(control_success, treatment_success, n_c, n_t):
    p_c = control_success / n_c
    p_t = treatment_success / n_t
    pooled = (control_success + treatment_success) / (n_c + n_t)
    se = math.sqrt(pooled * (1.0 - pooled) * (1.0 / n_c + 1.0 / n_t))
    lift = p_t - p_c
    z = lift / se
    relative_lift = lift / p_c
    return {"p_c": p_c, "p_t": p_t, "pooled": pooled, "se": se, "lift": lift, "relative_lift": relative_lift, "z": z}


def experiment_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    specs = [
        {"rung": "D1", "name": "hand 20-row randomized table", "n_c": 10, "n_t": 10, "p_c": 0.40, "p_t": 0.50, "latency_ms": 90},
        {"rung": "D2", "name": "20k clean experiment rows", "n_c": 10000, "n_t": 10000, "p_c": 0.048, "p_t": 0.052, "latency_ms": 110},
        {"rung": "D3", "name": "peeking and guardrail spike", "n_c": 25000, "n_t": 25000, "p_c": 0.050, "p_t": 0.054, "latency_ms": 230},
        {"rung": "D4", "name": "daily experiment trace", "n_c": 70000, "n_t": 70000, "p_c": 0.051, "p_t": 0.053, "latency_ms": 140},
        {"rung": "D5", "name": "multi-day production experiment", "n_c": 250000, "n_t": 250000, "p_c": 0.050, "p_t": 0.0525, "latency_ms": 180},
    ]
    workloads = []
    for spec in specs:
        c = local_rng.binomial(1, spec["p_c"], size=spec["n_c"])
        t = local_rng.binomial(1, spec["p_t"], size=spec["n_t"])
        guardrail = local_rng.normal(spec["latency_ms"], 15.0, size=spec["n_t"])
        workloads.append({**spec, "control": c, "treatment": t, "guardrail_latency": guardrail})
    return workloads


def peeking_false_positive_rate(simulations=400, looks=20, n_per_look=400, alpha_z=1.96):
    false_peek = 0
    false_fixed = 0
    alpha_spend_boundary = 2.8
    false_spend = 0
    local_rng = np.random.default_rng(SEED + 99)
    for _ in range(simulations):
        c = local_rng.binomial(1, 0.05, size=looks * n_per_look)
        t = local_rng.binomial(1, 0.05, size=looks * n_per_look)
        peek_hit = False
        spend_hit = False
        for look in range(1, looks + 1):
            end = look * n_per_look
            stats = ab_test_z(np.sum(c[:end]), np.sum(t[:end]), end, end)
            if abs(stats["z"]) > alpha_z:
                peek_hit = True
            if abs(stats["z"]) > alpha_spend_boundary:
                spend_hit = True
        final_stats = ab_test_z(np.sum(c), np.sum(t), c.size, t.size)
        false_peek += bool(peek_hit)
        false_fixed += bool(abs(final_stats["z"]) > alpha_z)
        false_spend += bool(spend_hit)
    return false_peek / simulations, false_fixed / simulations, false_spend / simulations


## The concept, built once: a randomized lift test

For binary outcomes, the lesson uses $z=(\hat p_T-\hat p_C)/\sqrt{\hat p(1-\hat p)(1/n_T+1/n_C)}$. Control has $480/10000=0.048$ and treatment has $520/10000=0.052$.

In [ ]:

stats = ab_test_z(480, 520, 10000, 10000)

assert stats["p_c"] == 0.048
assert stats["p_t"] == 0.052
assert round(stats["lift"], 3) == 0.004
print("control rate", stats["p_c"])
print("treatment rate", stats["p_t"])
print("absolute lift", stats["lift"])


The same calculation gives relative lift, pooled rate, standard error, and z. The lesson numbers are $0.004/0.048=0.083$, pooled rate $0.050$, standard error about $0.00308$, and z about $1.298$.

In [ ]:

relative_lift = stats["relative_lift"]
pooled = stats["pooled"]
se = stats["se"]
z = stats["z"]

assert round(relative_lift, 3) == 0.083
assert pooled == 0.05
assert round(se, 5) == 0.00308
assert round(z, 3) == 1.298
print("relative lift", relative_lift)
print("pooled rate", pooled)
print("standard error", se)
print("z score", z)


## The dataset ladder
D1-D5 move from a hand randomized table to a production multi-day experiment simulation.

In [ ]:

workloads = experiment_ladder()
for workload in workloads:
    preview = pd.DataFrame({
        "control": workload["control"][:8],
        "treatment": workload["treatment"][:8],
        "guardrail_latency_ms": np.round(workload["guardrail_latency"][:8], 1),
    })
    print(workload["rung"], workload["name"], "n_c=", workload["n_c"], "n_t=", workload["n_t"])
    print(preview)


In [ ]:

rows = []
outputs = []
for workload in workloads:
    result = ab_test_z(np.sum(workload["control"]), np.sum(workload["treatment"]), workload["n_c"], workload["n_t"])
    outputs.append((workload, result))
    rows.append({
        "rung": workload["rung"],
        "workload": workload["name"],
        "metric_z_statistic": float(result["z"]),
        "absolute_lift": float(result["lift"]),
        "standard_error": float(result["se"]),
        "p95_guardrail_ms": float(np.percentile(workload["guardrail_latency"], 95)),
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, (workload, result) in enumerate(outputs):
    axes[0, i].bar(["control", "treatment"], [result["p_c"], result["p_t"]], color=["#4c78a8", "#f58518"])
    axes[0, i].set_title(workload["rung"])
    axes[0, i].set_ylabel("conversion rate")
    panel_values = [result["z"], result["lift"] * 1000.0, np.percentile(workload["guardrail_latency"], 95)]
    axes[1, i].bar(["z", "lift x1000", "p95 ms"], panel_values, color=["#e45756", "#54a24b", "#b279a2"])
    axes[1, i].set_title("operational panel")
summary_x = np.arange(len(metrics))
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary_x, metrics["metric_z_statistic"], marker="o", label="z statistic")
ax.plot(summary_x, metrics["absolute_lift"], marker="s", label="absolute lift")
ax.axhline(1.96, color="#e45756", linestyle="--", label="two-sided 5%")
ax.set_xticks(summary_x)
ax.set_xticklabels(metrics["rung"])
ax.set_title("z and lift vs sample size")
ax.set_xlabel("D1 to D5 sample size")
ax.legend()
plt.show()


## Pitfall on D5: peeking until significant
Repeated looks change the stopping rule and inflate false positives. A fixed horizon or stricter alpha-spending boundary keeps the error rate closer to the design.

In [ ]:

peek_rate, fixed_rate, spend_rate = peeking_false_positive_rate()
print("false positive rate with peeking", peek_rate)
print("false positive rate fixed horizon", fixed_rate)
print("false positive rate alpha spending", spend_rate)


## Evaluate it + practice

- Metric: standard-error-adjusted lift as a z statistic, with latency guardrails.
- No-skill baseline: compare before-after metrics without randomization.
- Cheap sanity check: swapping labels should flip the sign of lift.
- Ablation: turn off pooling or randomization and z becomes misleading.
- Failure signals: peeking, underpowered tests, guardrail regressions, or segment imbalance.

Practice: Compute the confidence interval for D2.

Practice: Add a guardrail fail column to the metrics table.

Practice: Simulate a smaller expected lift and inspect the standard error.